# Classification Probability Layers

Ensemble posterior、calibration、reliability、conformalの責務をprobability-spaceで整理します。


In [ ]:
import torch

from robotorchan.models.classification import (
    ClassificationEnsemblePosterior,
    TemperatureScalingCalibrator,
    classification_nll,
)

torch.set_default_dtype(torch.double)
torch.manual_seed(0)


## 1. 小さな合成データ

In [ ]:
train_X = torch.tensor([[0.05], [0.15], [0.30], [0.48], [0.52], [0.70], [0.85], [0.95]])
train_Y = (train_X.squeeze(-1) > 0.5).to(dtype=torch.double)
grid = torch.linspace(0.0, 1.0, 51).unsqueeze(-1)


## 2. Probability ensemble posterior

In [ ]:
p = grid.squeeze(-1)
members = torch.stack([
    torch.stack((1.0 - p, p), dim=-1),
    torch.stack((1.0 - p.pow(0.8), p.pow(0.8)), dim=-1),
    torch.stack((1.0 - p.pow(1.2), p.pow(1.2)), dim=-1),
])
posterior = ClassificationEnsemblePosterior(members)
print(posterior.mean.shape)
print(posterior.probability_variance.shape)


## 3. Probability calibration

In [ ]:
raw = torch.tensor([[0.90, 0.10], [0.65, 0.35], [0.30, 0.70], [0.05, 0.95]])
targets = torch.tensor([0, 0, 1, 1])
calibrator = TemperatureScalingCalibrator()
calibrator.fit(raw, targets)
calibrated = calibrator(raw)
print(classification_nll(raw, targets).item())
print(classification_nll(calibrated, targets).item())


## 4. Reliability / conformalとの違い

Reliability/OODはentropy・disagreement・input distanceを分離して診断します。Split conformalはheld-out calibration setからprediction setを構成します。probability calibrationとは別レイヤーです。

実運用ではmodel fitting、probability calibration、conformal calibrationに使うデータの役割を明示してください。